# 第15章 元学习（Meta Learning）

> "学会如何学习——让机器自己找出最优的学习方法、模型架构和初始化参数。"

## 1. 知识地图：章节结构与概览

```
15.1 元学习的概念
    ├── 什么是"学习如何学习"
    ├── 元学习的基本框架
    ├── 元学习函数 F 的定义
    └── 与一般机器学习的区别
15.2 元学习的三个步骤
    ├── 步骤一：定义可学习的学习算法 F_φ
    │   ├── 学习网络架构
    │   ├── 学习初始化参数
    │   └── 学习学习率等超参数
    ├── 步骤二：定义损失函数 L(φ)
    │   ├── 训练任务的收集
    │   ├── 多任务损失的聚合
    │   └── 测试数据在训练过程中的使用
    └── 步骤三：优化算法寻找最优 φ*
        ├── 梯度下降法
        ├── 强化学习
        └── 进化算法
15.3 元学习与机器学习的对比
    ├── 目标差异
    ├── 训练数据差异（支持集 vs 查询集）
    ├── 训练框架差异（跨任务 vs 单一任务）
    ├── 内循环与外循环
    ├── 共同之处：过拟合、数据增强、验证集
    └── 回合（Episode）的概念
15.4 元学习的实例算法
    ├── MAML（模型诊断元学习）
    │   ├── 学习最优初始化参数
    │   ├── 两次梯度计算过程
    │   ├── 与自监督学习的对比
    │   └── 变体：ANIL, FOMAML, Reptile
    ├── 可学习的优化器
    │   └── "Learning to learn by gradient descent by gradient descent"
    ├── 神经网络架构搜索（NAS）
    │   ├── 强化学习方法
    │   ├── 进化算法
    │   └── 可微分架构搜索
    ├── 可学习的数据处理
    │   ├── 自动数据增强
    │   └── 自动样本权重
    └── Learning to Compare（度量学习方法）
15.5 元学习的应用
    ├── 少样本图像分类（Few-shot Classification）
    ├── N-way K-shot 任务设定
    └── Omniglot 基准数据集
```

## 2. 核心概念：什么是元学习？

### 2.1 直觉类比

想象你是一个老师，你不仅要教会学生解一道数学题，你还要教会学生们"如何高效学习数学"的方法论。这个"关于学习方法的学习"就是元学习的核心思想。

### 2.2 形式化定义

元学习的目标是找到一个函数 $F$（即学习算法），使得对于任意给定的训练数据集，$F$ 能够输出一个在该任务上表现良好的模型 $f$：

$$F(\text{训练数据}) = f$$

其中 $f$ 本身是一个函数（如分类器），输入为测试样本 $x$，输出为预测结果 $\hat{y}$。

**元学习要优化的目标：**

$$\phi^* = \arg\min_\phi L(\phi)$$

其中 $L(\phi)$ 是所有训练任务上的总损失：

$$L(\phi) = \sum_{n=1}^{N} l_n$$

这里 $l_n$ 是第 $n$ 个任务中的**测试损失**（不是训练损失！）。

### 2.3 为什么需要元学习？

传统深度学习的困境：
- 调超参数（学习率、网络结构、优化器选择）需要人工反复试验
- 工业界买大量 GPU 同时跑多组参数
- 学术界GPU有限，只能凭经验猜参数
- 元学习让机器**自己学会**如何调参和设计网络结构

## 3. 元学习的三个步骤（详析）

### 步骤一：定义可学习的学习算法 $F_\phi$

与机器学习中学习参数 $\theta$ 类似，元学习中要学习算法本身的参数 $\phi$。可学习的成分包括：

| 可学习的成分 | 说明 | 对应的元学习方法 |
|------|------|------|
| 网络初始化参数 $\theta_0$ | 让模型有一个好的起点 | MAML, Reptile |
| 优化器超参数（学习率等） | 自动选择更新策略 | Learning to Optimize |
| 网络架构 | 自动设计网络结构 | NAS |
| 数据处理策略 | 自动决定数据增强和样本权重 | 可学习数据处理 |

### 步骤二：定义损失函数 $L(\phi)$

损失函数衡量学习算法的好坏。做法：
1. 收集多个训练任务，每个任务包含训练数据（支持集 Support Set）和测试数据（查询集 Query Set）
2. 对每个任务，用支持集训练模型得到 $f_{\theta_n^*}$，再用查询集评估性能
3. 将所有任务上的查询集损失求和得到 $L(\phi)$

**关键：元学习中损失函数是用测试数据（查询集）计算的，而传统机器学习中用训练数据计算。**

### 步骤三：优化算法寻找最优 $\phi^*$

- **梯度下降法**：当 $L(\phi)$ 可微时最常用
- **强化学习**：当无法计算梯度时（如NAS），将 $-L(\phi)$ 作为奖励
- **进化算法**：另一种处理不可微问题的方法

## 4. 元学习 vs 机器学习：完整对比

| 维度 | 机器学习 | 元学习 |
|------|------|------|
| **目标** | 找一个函数 $f$（如分类器） | 找一个学习算法 $F$（函数产生器） |
| **训练单元** | 训练/测试样本 | 训练/测试任务 |
| **训练数据名称** | 训练数据 | 支持集（Support Set） |
| **测试数据名称** | 测试数据 | 查询集（Query Set） |
| **损失计算** | 用训练数据计算 | 用查询集（测试数据）计算 |
| **学习的称呼** | 单一任务学习 | 跨任务学习 |
| **测试的称呼** | 单一任务测试 | 跨任务测试 |
| **循环结构** | 一个任务内训练+测试 | 外循环（跨任务）包含多次内循环 |

**内循环和外循环：**
- **内循环（Inner Loop）**：单个任务的训练+测试过程
- **外循环（Outer Loop）**：对所有任务进行循环，更新 $\phi$（跨任务训练）

**共同之处：**
- 元学习也会遇到过拟合（在训练任务上表现好，测试任务上表现差）
- 解决方案类似：收集更多训练任务、任务级数据增强
- 同样需要验证集（验证任务）来选择超参数

## 5. MAML 算法详解

### 5.1 MAML 的核心思想

MAML (Model-Agnostic Meta-Learning) 学习的是**最优的模型初始化参数** $\theta_0$，使得模型在每个新任务上只需几步梯度下降就能达到很好的性能。

**直觉类比：** 找到一个"万能起点"，从该起点出发，对任何任务只需微调几步就能达到好效果。就像找到一个在所有数学题型上都有良好初始直觉的学生。

### 5.2 MAML 的两层优化

**内循环（Inner Loop）—— 任务适应：**

$$\theta_i' = \theta - \alpha \nabla_\theta \mathcal{L}_{\mathcal{T}_i}^{\text{support}}(f_\theta)$$

- $\alpha$：内循环学习率
- $\mathcal{L}_{\mathcal{T}_i}^{\text{support}}$：任务 $i$ 在支持集上的损失

**外循环（Outer Loop）—— 元更新：**

$$\theta \leftarrow \theta - \beta \nabla_\theta \sum_i \mathcal{L}_{\mathcal{T}_i}^{\text{query}}(f_{\theta_i'})$$

- $\beta$：外循环学习率
- 注意：外循环梯度要穿过内循环的更新过程，即计算"梯度的梯度"（二阶导数）

### 5.3 MAML 变体

- **FOMAML（First-Order MAML）**：忽略二阶导数项，近似为一阶梯度
- **ANIL（Almost No Inner Loop）**：只在最后一层做内循环更新
- **Reptile**：简化版 MAML，不做二次梯度，将每个任务的方向作为更新方向

In [ ]:
# MAML 的核心 PyTorch 实现
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict
import copy

class MAMLModel(nn.Module):
    """用于 MAML 的简单分类网络"""
    def __init__(self, input_dim=28*28, hidden_dim=64, output_dim=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

class MAML:
    """MAML 算法实现"""
    def __init__(self, model, inner_lr=0.01, outer_lr=0.001, inner_steps=5):
        self.model = model
        self.inner_lr = inner_lr     # 内循环学习率 α
        self.outer_lr = outer_lr     # 外循环学习率 β
        self.inner_steps = inner_steps
        self.meta_optimizer = torch.optim.Adam(self.model.parameters(), lr=outer_lr)
    
    def inner_loop(self, support_x, support_y):
        """内循环：在单个任务的支持集上做 K 步梯度下降"""
        fast_weights = OrderedDict(self.model.named_parameters())
        
        for _ in range(self.inner_steps):
            logits = self._forward(support_x, fast_weights)
            loss = F.cross_entropy(logits, support_y)
            grads = torch.autograd.grad(loss, fast_weights.values(), create_graph=True)
            fast_weights = OrderedDict(
                (name, param - self.inner_lr * grad)
                for (name, param), grad in zip(fast_weights.items(), grads)
            )
        return fast_weights
    
    def outer_loop(self, tasks_batch):
        """外循环：在批量任务上进行元更新"""
        meta_loss = 0.0
        for support_x, support_y, query_x, query_y in tasks_batch:
            fast_weights = self.inner_loop(support_x, support_y)
            logits = self._forward(query_x, fast_weights)
            task_loss = F.cross_entropy(logits, query_y)
            meta_loss += task_loss
        self.meta_optimizer.zero_grad()
        meta_loss.backward()
        self.meta_optimizer.step()
        return meta_loss.item()
    
    def _forward(self, x, weights):
        x = F.linear(x, weights['net.0.weight'], weights['net.0.bias'])
        x = F.relu(x)
        x = F.linear(x, weights['net.2.weight'], weights['net.2.bias'])
        x = F.relu(x)
        x = F.linear(x, weights['net.4.weight'], weights['net.4.bias'])
        return x

print("MAML 核心类定义完成")
print("关键：外循环梯度穿过内循环更新（二阶导数）")

In [ ]:
# FOMAML（一阶MAML）的简化实现
import torch
import torch.nn.functional as F

class FOMAML:
    """First-Order MAML：忽略二阶导数，计算效率更高"""
    def __init__(self, model, inner_lr=0.01, outer_lr=0.001, inner_steps=5):
        self.model = model
        self.inner_lr = inner_lr
        self.outer_lr = outer_lr
        self.inner_steps = inner_steps
        self.meta_optimizer = torch.optim.Adam(self.model.parameters(), lr=outer_lr)
    
    def train_on_batch(self, tasks_batch):
        """FOMAML 关键：内循环不保留二阶计算图"""
        meta_loss = 0.0
        for support_x, support_y, query_x, query_y in tasks_batch:
            original_params = [p.clone() for p in self.model.parameters()]
            # 内循环（不保留二阶计算图）
            for _ in range(self.inner_steps):
                logits = self.model(support_x)
                inner_loss = F.cross_entropy(logits, support_y)
                grads = torch.autograd.grad(inner_loss, self.model.parameters())
                for p, g in zip(self.model.parameters(), grads):
                    p.data = p.data - self.inner_lr * g
            # 外循环用查询集
            logits = self.model(query_x)
            task_loss = F.cross_entropy(logits, query_y)
            meta_loss += task_loss
            # 恢复原始参数
            for p, orig in zip(self.model.parameters(), original_params):
                p.data = orig
        self.meta_optimizer.zero_grad()
        meta_loss.backward()
        self.meta_optimizer.step()
        return meta_loss.item()

print("FOMAML 核心类定义完成")
print("与MAML的关键区别：内循环不保留计算图(create_graph=False)")

## 6. 神经网络架构搜索（NAS）

### 6.1 NAS 的三种方法

**1. 基于强化学习的 NAS：**
- 用一个 RNN 控制器作为智能体
- 控制器输出网络架构参数（滤波器大小、步长等）
- 训练该架构，用验证集准确率作为奖励 $R = -L(\phi)$
- 用策略梯度更新控制器

**2. 基于进化算法的 NAS：**
- 维护一个网络架构的种群
- 通过变异和交叉产生新架构
- 选择性能最好的保留

**3. 可微分架构搜索（DARTS）：**
- 将离散的架构选择转化为连续的可微分参数
- 可以直接用梯度下降法同时优化架构参数和网络权重
- 计算效率远高于前两种

In [ ]:
# Prototypical Network —— 基于度量的元学习
import torch
import torch.nn as nn
import torch.nn.functional as F

class PrototypicalNetwork(nn.Module):
    """
    原型网络 - Learning to Compare 方法
    核心思想：
    1. 用支持集计算每个类别的"原型"（类中心向量）
    2. 查询样本分类到距离最近的原型
    不需要梯度下降内循环！
    """
    def __init__(self, input_dim=28*28, embedding_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, embedding_dim)
        )
    
    def forward(self, support_x, support_y, query_x, n_way):
        support_emb = self.encoder(support_x)
        query_emb = self.encoder(query_x)
        # 计算原型
        prototypes = []
        for c in range(n_way):
            class_embeddings = support_emb[support_y == c]
            prototypes.append(class_embeddings.mean(dim=0))
        prototypes = torch.stack(prototypes)
        # 距离计算
        dists = torch.cdist(query_emb, prototypes)
        logits = -dists  # 距离越小分数越高
        return logits

print("Prototypical Network 定义完成")
print("优点：不需要内循环梯度更新，直接用度量比较")

In [ ]:
# N-way K-shot 任务采样器
import numpy as np
import torch

class FewShotTaskSampler:
    """为元学习创建 N-way K-shot 任务的采样器"""
    def __init__(self, dataset, n_way=5, k_shot=1, n_query=15, n_tasks=100):
        self.dataset = dataset
        self.n_way = n_way
        self.k_shot = k_shot
        self.n_query = n_query
        self.n_tasks = n_tasks
        self.labels = np.array([label for _, label in dataset])
        self.unique_labels = np.unique(self.labels)
        self.label_to_indices = {
            label: np.where(self.labels == label)[0]
            for label in self.unique_labels
        }

    def sample_task(self):
        """采样一个 N-way K-shot 任务"""
        task_labels = np.random.choice(self.unique_labels, self.n_way, replace=False)
        support_x, support_y = [], []
        query_x, query_y = [], []
        for new_label, orig_label in enumerate(task_labels):
            indices = self.label_to_indices[orig_label]
            sampled = np.random.choice(indices, self.k_shot + self.n_query, replace=False)
            for idx in sampled[:self.k_shot]:
                support_x.append(self.dataset[idx][0])
                support_y.append(new_label)
            for idx in sampled[self.k_shot:]:
                query_x.append(self.dataset[idx][0])
                query_y.append(new_label)
        return (
            torch.stack(support_x), torch.tensor(support_y),
            torch.stack(query_x), torch.tensor(query_y)
        )
    
    def sample_batch(self):
        return [self.sample_task() for _ in range(self.n_tasks)]

print("任务采样器定义完成")
print("用途：从Omniglot等基准数据集中构造N-way K-shot任务")

## 7. 常见误区与易错点

### 误区 1：元学习 = 小样本学习
**纠正：** 小样本学习是目标（场景），元学习是方法。还有其他方法（如迁移学习、数据增强）也能做小样本学习。

### 误区 2：元学习不需要调超参数
**纠正：** 元学习本身也有大量超参数需要调整（外循环学习率、内循环学习率、内循环步数等）。

### 误区 3：用训练任务的测试数据做训练是数据泄露
**纠正：** 在元学习中，训练任务和测试任务是**不重叠**的（不同类别/不同域）。训练任务中的查询集是用来评估学习算法好坏的，不是最终应用的数据。

### 误区 4：元学习 = 域适应/迁移学习
**纠正：** 元学习更强调"学习如何快速适应"这个能力本身，而不是单纯的知识迁移。

### 误区 5：MAML 总是比预训练好
**纠正：** 在某些场景下，用大量数据预训练（如自监督学习）比 MAML 用大量任务预训练更有效。两者是互补的。

## 8. 与其他章节的联系

| 章节 | 联系 |
|------|------|
| 第3章 深度学习基础（梯度下降） | MAML 的核心操作就是"梯度下降的梯度下降" |
| 第8章 GAN | GAN 的训练过程也是一种双循环 |
| 第10章 自监督学习（BERT/预训练） | 都致力于找好的初始化参数，方法不同 |
| 第16章 终身学习 | 多任务场景的处理——元学习重"快速适应"，终身学习重"不忘旧知识" |
| 第17章 网络压缩（彩票假说） | 彩票假说和 MAML 都关心初始化参数的重要性 |
| 第19章 ChatGPT（预训练） | "预训练+微调"与 MAML 的"学初始化+快速适应"有概念共性 |

## 9. 核心公式汇总

### 元学习损失函数

$$L(\phi) = \sum_{n=1}^{N} l_n$$

- $\phi$：学习算法的参数；$N$：训练任务总数；$l_n$：第 $n$ 个任务的查询集损失

### 机器学习损失函数（对比）

$$L(\theta) = \sum_{k=1}^{K} e_k$$

- $\theta$：模型参数；$K$：一个任务中训练样本总数；$e_k$：第 $k$ 个训练样本的损失

### MAML 内循环更新

$$\theta_i' = \theta - \alpha \nabla_\theta \mathcal{L}_{\mathcal{T}_i}^{\text{support}}(f_\theta)$$

### MAML 外循环更新

$$\theta \leftarrow \theta - \beta \nabla_\theta \sum_i \mathcal{L}_{\mathcal{T}_i}^{\text{query}}(f_{\theta_i'})$$

### NAS 优化目标（强化学习形式）

$$\phi^* = \arg\max_\phi \mathbb{E}[R(\text{网络架构})]$$

其中 $R$ 是使用该架构时的验证准确率（即 $-L(\phi)$）。

## 10. 关键总结

1. **元学习是"学习如何学习"**：目标不是直接找一个分类器，而是找一个能根据训练数据快速产生分类器的学习算法
2. **训练单位是任务，而不是样本**：元学习的训练数据是一个个"分类任务"，而非单个图片
3. **损失计算在查询集上**：这是元学习与传统机器学习最本质的区别之一
4. **内循环 + 外循环是核心结构**：外循环（跨任务）优化学习算法参数，内循环（单任务训练）产生具体模型
5. **MAML 学习最优初始化参数**：目标是"万能起点"，对任何新任务只需几步梯度下降就能适应
6. **Learning to Compare 抛弃了梯度下降**：直接用网络处理整个训练+测试流程
7. **元学习与自监督学习/预训练有深层联系**：都试图找到好的初始化参数，但途径不同
8. **元学习也会过拟合**：需要收集更多训练任务或做任务级数据增强
9. **NAS 是元学习的重要应用**：自动搜索最优网络架构
10. **N-way K-shot 是标准基准**：Omniglot 数据集是元学习研究的标志性基准